In [1]:
#!pip install folium

In [1]:
import osmnx as ox
import matplotlib.pyplot as plt
import pandas as pd

import folium

ox.settings.overpass_rate_limit = False  # skip server status check
ox.settings.use_cache = True

ox.settings.overpass_url = 'https://overpass.kumi.systems/api'

# Outcome Target:

Identify dozens of nearby businesses to begin my GrooveSeeker outreach. I will target nearby cities/towns. I want to capture their business/org name, type, and contact information. I want to produce a dataframe and then a spreadsheet that I will use to prioritize and then do my outreach.

# Methodology: 

- Identify city pindrops for each target location based on nearest MAX train station
- Identify all businesses/orgs within a one mile redius from each pindrop;
- Put them on a map, too so that I can explore the map and plan my routes

I will target places that are probably hosting events. I will aim to capture a lot and then delete anything from the spreadsheet that is not helpful. In this notebook, I want to cast a wide net. It is easy enough to clean a spreadsheet.

# Goals:

- Simple spreadsheet
    - Nearest City
    - Business/Organization Name
    - Contact Information
    
# OSINT Techniques
- Natural Language Processing
- Spatial Analysis
- Data Science

# First, Create the Pindrop Dictionary

In [2]:
pindrops = {
    'forest_grove': {
        'address': '2004 Main St, Forest Grove, OR 97116',
        'city': 'Forest Grove'
    },
    'hillsboro': {
        'address': '333 SE Washington St, Hillsboro, OR 97123',
        'city': 'Hillsboro'
    },
    'orenco': {
        'address': '6199 NE Alder St, Hillsboro, OR 97124',
        'city': 'Hillsboro'
    },
    'beaverton': {
        'address': '4050 SW Lombard Ave, Beaverton, OR 97005',
        'city': 'Beaverton'
    },
    'pdx_west': {
        'address': '1844 SW Morrison St, Portland, OR 97205',
        'city': 'Portland'
    },
    'pdx_north': {
        'address': '1313 W Burnside St, Portland, OR 97209',
        'city': 'Portland'
    },
    'pdx_east': {
        'address': '2724 E Burnside St, Portland, OR 97214',
        'city': 'Portland'
    }
}

pindrops.keys()

dict_keys(['forest_grove', 'hillsboro', 'orenco', 'beaverton', 'pdx_west', 'pdx_north', 'pdx_east'])

In [3]:
def get_local_leads(address, city, max_distance):

    tags = {
        'amenity': True,
        'shop': True,
        'tourism': True,
        'leisure': True,
        'office': True,
        'craft': True,
        'club': True,
        'historic': True
    }

    businesses = ox.features_from_address(address, tags, dist=max_distance)
    businesses_filtered = businesses.dropna(subset=['name'])

    results = []

    for idx, row in businesses_filtered.iterrows():

        entity_type = None

        for tag in ['amenity', 'shop', 'tourism', 'leisure', 'office', 'craft', 'club', 'historic']:

            value = row.get(tag)

            if pd.notna(value):
                entity_type = value
                break

        street = row.get('addr:street')
        number = row.get('addr:housenumber')

        if pd.notna(street) and pd.notna(number):
            business_address = '{} {}'.format(number, street)
        elif pd.notna(street):
            business_address = street
        else:
            business_address = None

        phone = row.get('phone')

        if pd.isna(phone):
            phone = None

        results.append({
            'business_name': row.get('name'),
            'type': entity_type,
            'nearest_city': city,
            'address': business_address,
            'phone_number': phone,
            'pindrop_address': address
        })

    print("Found {} businesses within {}m".format(len(results), max_distance))

    return results

In [4]:
def draw_local_leads(address, city, max_distance):

    tags = {
        'amenity': True,
        'shop': True,
        'tourism': True,
        'leisure': True,
        'office': True,
        'craft': True,
        'club': True,
        'historic': True
    }

    businesses = ox.features_from_address(address, tags, dist=max_distance)
    businesses_filtered = businesses.dropna(subset=['name'])

    address_coords = ox.geocode(address)

    m = folium.Map(
        location=address_coords,
        zoom_start=16,
        tiles='OpenStreetMap'
    )

    folium.CircleMarker(
        address_coords,
        radius=10,
        popup='<b>{}</b><br>{}'.format(city, address),
        color='#2E86DE',
        fill=True,
        fillColor='#2E86DE',
        fillOpacity=1.0,
        weight=3
    ).add_to(m)

    for idx, row in businesses_filtered.iterrows():

        if 'geometry' in row and row['geometry']:

            coords = [
                row['geometry'].centroid.y,
                row['geometry'].centroid.x
            ]

            entity_type = None

            for tag in ['amenity', 'shop', 'tourism', 'leisure', 'office', 'craft', 'club', 'historic']:

                value = row.get(tag)

                if pd.notna(value):
                    entity_type = value
                    break

            name = row.get('name', 'Unknown')

            folium.CircleMarker(
                coords,
                radius=6,
                popup='<b>{}</b><br>{}'.format(name, entity_type),
                color='#EE5A6F',
                fill=True,
                fillColor='#EE5A6F',
                fillOpacity=0.8
            ).add_to(m)

    return m

In [5]:
max_distance = 1000 # global value

In [6]:
export_data = []

# Forest Grove

In [7]:
address = pindrops['forest_grove']['address']
city = pindrops['forest_grove']['city']

leads = get_local_leads(address, city, max_distance)
export_data.append(leads)
leads[0] # preview

Found 110 businesses within 1000m


{'business_name': 'Fitness 1440',
 'type': 'sports_centre',
 'nearest_city': 'Forest Grove',
 'address': '2329 Pacific Avenue',
 'phone_number': '+1-503-746-7819',
 'pindrop_address': '2004 Main St, Forest Grove, OR 97116'}

In [8]:
draw_local_leads(address, city, max_distance)

# Hillsboro

In [9]:
address = pindrops['hillsboro']['address']
city = pindrops['hillsboro']['city']

leads = get_local_leads(address, city, max_distance)
export_data.append(leads)
leads[0] # preview

Found 295 businesses within 1000m


{'business_name': "Wilson's Cafe & Sports Bar",
 'type': 'pub',
 'nearest_city': 'Hillsboro',
 'address': '134 Northeast 2nd Avenue',
 'phone_number': None,
 'pindrop_address': '333 SE Washington St, Hillsboro, OR 97123'}

In [10]:
draw_local_leads(address, city, max_distance)

# Orenco

In [11]:
address = pindrops['orenco']['address']
city = pindrops['orenco']['city']

leads = get_local_leads(address, city, max_distance)
export_data.append(leads)
leads[0] # preview

Found 99 businesses within 1000m


{'business_name': 'Swagath',
 'type': 'restaurant',
 'nearest_city': 'Hillsboro',
 'address': None,
 'phone_number': None,
 'pindrop_address': '6199 NE Alder St, Hillsboro, OR 97124'}

In [12]:
draw_local_leads(address, city, max_distance)

# Beaverton

In [13]:
address = pindrops['beaverton']['address']
city = pindrops['beaverton']['city']

leads = get_local_leads(address, city, max_distance)
export_data.append(leads)
leads[0] # preview

Found 623 businesses within 1000m


{'business_name': 'Book Corner',
 'type': 'books',
 'nearest_city': 'Beaverton',
 'address': '12470 Southwest 5th Street',
 'phone_number': None,
 'pindrop_address': '4050 SW Lombard Ave, Beaverton, OR 97005'}

In [14]:
draw_local_leads(address, city, max_distance)

# PDX West

In [15]:
address = pindrops['pdx_west']['address']
city = pindrops['pdx_west']['city']

leads = get_local_leads(address, city, max_distance)
export_data.append(leads)
leads[0] # preview

Found 1208 businesses within 1000m


{'business_name': 'Pod',
 'type': 'artwork',
 'nearest_city': 'Portland',
 'address': None,
 'phone_number': None,
 'pindrop_address': '1844 SW Morrison St, Portland, OR 97205'}

In [16]:
draw_local_leads(address, city, max_distance)

# PDX North

In [17]:
address = pindrops['pdx_north']['address']
city = pindrops['pdx_north']['city']

leads = get_local_leads(address, city, max_distance)
export_data.append(leads)
leads[0] # preview

Found 1767 businesses within 1000m


{'business_name': 'Pod',
 'type': 'artwork',
 'nearest_city': 'Portland',
 'address': None,
 'phone_number': None,
 'pindrop_address': '1313 W Burnside St, Portland, OR 97209'}

In [18]:
draw_local_leads(address, city, max_distance)

# PDX East

In [19]:
address = pindrops['pdx_east']['address']
city = pindrops['pdx_east']['city']

leads = get_local_leads(address, city, max_distance)
export_data.append(leads)
leads[0] # preview

Found 327 businesses within 1000m


{'business_name': 'Kerr Youth Center at Wynne Watts School',
 'type': 'school',
 'nearest_city': 'Portland',
 'address': '424 Northeast 22nd Avenue',
 'phone_number': None,
 'pindrop_address': '2724 E Burnside St, Portland, OR 97214'}

In [20]:
draw_local_leads(address, city, max_distance)

# DataFrame Exploration

In [21]:
export_data_flat = [
    lead
    for leads in export_data
    for lead in leads
]

df = pd.DataFrame(export_data_flat)

df.to_csv('local_leads.csv', index=False)

df.head()

,business_name,type,nearest_city,address,phone_number,pindrop_address
0,Fitness 1440,sports_centre,Forest Grove,2329 Pacific Avenue,+1-503-746-7819,"2004 Main St, Forest Grove, OR 97116"
1,Pizza Hut,restaurant,Forest Grove,2323 Pacific Avenue,+1 503-357-3368,"2004 Main St, Forest Grove, OR 97116"
2,Cornerstone Pub & Grill,restaurant,Forest Grove,2307 Pacific Avenue,+1-503-357-4742,"2004 Main St, Forest Grove, OR 97116"
3,La Sierra,restaurant,Forest Grove,2331 Pacific Avenue,+1-503-992-8980,"2004 Main St, Forest Grove, OR 97116"
4,Tuality Physical Therapy,physical_therapist,Forest Grove,2333 Pacific Avenue,None,"2004 Main St, Forest Grove, OR 97116"


In [22]:
df.shape 

# this many potential leads found; i cast a wide net; even 50% signal/noise is easy for me to work with
# i do not mind manually cleaning up spreadsheets. that's easy. :)

(4429, 6)